# Train your own probe with Undercurrent

*See the undercurrent before it surfaces.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wrynx/undercurrent/blob/main/notebooks/train_probe.ipynb)

A **probe** is a small classifier that reads a language model's internal activations. In this
notebook you build one end to end, on a CPU, in a few minutes:

1. take a labelled text dataset (toxic vs. non-toxic comments),
2. **collect** one residual-stream activation per example with Undercurrent,
3. **train** a logistic-regression probe (and a small MLP) on those vectors and evaluate it,
4. **save** the weights with `safetensors` and load them back into a fresh probe,
5. run the trained probe **live** while the model generates, so it can stop a generation.

The notebook reuses the code of the repository's training example,
[`examples/train_probe/`](https://github.com/wrynx/undercurrent/tree/main/examples/train_probe),
so the two always do the same thing. That example also has a command-line script for bigger runs on a GPU.

## 1. Install

`pip install undercurrent` brings PyTorch, Transformers and the Hugging Face backend. The training
helpers are example code that isn't part of the installed package, so we also fetch the repository
(shallow clone), and install `datasets` to download the labelled data.

In [ ]:
%pip install -q undercurrent "datasets>=2.14"
# Or the latest development version from GitHub `main`:
# %pip install -q "undercurrent @ git+https://github.com/wrynx/undercurrent.git@main" "datasets>=2.14"
!test -d undercurrent || git clone -q --depth 1 https://github.com/wrynx/undercurrent.git

## 2. Configuration

Everything you might want to change is here. Each value can also be set with an environment
variable (the automated test uses that to swap in a tiny random model and a handful of examples).

- `MODEL`: any decoder-only Hugging Face model; `gpt2` runs comfortably on a CPU.
- `LAYER`: the layer to read. A middle layer is a good default: probing work generally finds
  high-level features most linearly readable there.
- `DATASET`: `civil_comments` (downloaded from the Hugging Face Hub) or `toy` (48 hand-written
  lines built in code, no download; useful offline, but far too small and templated to learn
  anything real).
- `N_EXAMPLES`: how many labelled examples to sample, half from each class. Collection is one
  forward pass per example, so the runtime grows linearly with it.

In [ ]:
import os
import sys
from pathlib import Path

import torch
from transformers import AutoConfig

MODEL = os.environ.get("UNDERCURRENT_NB_MODEL", "gpt2")
DATASET = os.environ.get("UNDERCURRENT_NB_DATASET", "civil_comments")
N_EXAMPLES = int(os.environ.get("UNDERCURRENT_NB_N_EXAMPLES", "200"))
LAYER = AutoConfig.from_pretrained(MODEL).num_hidden_layers // 2  # 6 for gpt2
POSITION = "prompt[-1]"  # the last prompt token: the probe judges the prompt before generation
EPOCHS = 20
SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Where the training example lives: the clone from the install cell, or this repository checkout.
candidates = [os.environ.get("UNDERCURRENT_EXAMPLES_DIR"), "undercurrent/examples", "../examples", "examples"]
EXAMPLE_DIR = next(
    (Path(c) / "train_probe" for c in candidates if c and (Path(c) / "train_probe" / "train_probe.py").is_file()),
    None,
)
if EXAMPLE_DIR is None:
    raise FileNotFoundError("examples/train_probe not found: run the install cell, or start Jupyter in a repo clone")
sys.path.insert(0, str(EXAMPLE_DIR.resolve()))

print(f"model={MODEL} layer={LAYER} dataset={DATASET} n_examples={N_EXAMPLES} device={DEVICE}")
print(f"training helpers from {EXAMPLE_DIR}")

## 3. The dataset

[`google/civil_comments`](https://huggingface.co/datasets/google/civil_comments) (CC0 1.0) holds
about 1.8M public comments, each rated by crowd workers with a `toxicity` score between 0 and 1.
The example's loader (`data_sources.load_examples`) turns that into a binary task:

- label **1, `toxic`**: `toxicity >= 0.5`, the class the probe should flag;
- label **0, `non_toxic`**: everything else.

It streams the train split (no full download), shuffles it with a fixed seed, keeps
`N_EXAMPLES // 2` comments per class (toxic comments are a small minority, so we balance them),
drops empty comments and cuts each to 1000 characters.

The data contains offensive text, and the labels carry the raters' biases; see the dataset card.

In [ ]:
from data_sources import load_examples

examples, label_names = load_examples(DATASET, N_EXAMPLES, SEED)
texts = [text for text, _ in examples]
print(f"{len(examples)} examples, labels: 0={label_names[0]!r}, 1={label_names[1]!r}")
print(f"class counts: {[sum(1 for _, y in examples if y == c) for c in (0, 1)]}")
for text, label in examples[:4]:
    print(f"  [{label_names[label]}] {text[:80]!r}")

## 4. Collect activations

Collection uses Undercurrent's normal capture path: a spec with one extraction point and a probe
that, instead of judging activations, just keeps a copy of them (`ActivationCollectorProbe` from
`collector.py`). `collect_activations` builds this spec:

```yaml
extraction_points:
  - name: collect
    layers: LAYER
    tensor_type: residual_stream
    position: "prompt[-1]"
    probe_type: activation_collector
    probe_kind: single_shot
```

and runs `ProbedModel.generate(text, max_new_tokens=1)` once per example. Each run returns one
vector of size `hidden_size` from the probe's verdict, and they are stacked into a feature matrix.

In [ ]:
from train_probe import collect_activations

features, kept = collect_activations(MODEL, texts, layer=LAYER, position=POSITION, device=DEVICE)
labels = [examples[i][1] for i in kept]
y = torch.tensor(labels, dtype=torch.float32)

print(f"features: {tuple(features.shape)} {features.dtype}  (examples x hidden size)")
print(f"labels:   {tuple(y.shape)}, {int(y.sum())} positive")

## 5. Train and evaluate

Split each class 70/15/15 into train, validation and test sets, then train with the example's
`train_head`: features are standardised with the training-set mean and standard deviation (stored
in the probe so it sees inputs on the same scale later), the loss is class-balanced binary
cross-entropy, and the epoch with the lowest validation loss is kept.

We train two probes: `linear` (logistic regression, `hidden_size + 1` parameters) and `mlp` (one
hidden layer of 64 units). Metrics come from the example's `binary_metrics`; AUROC is computed with
numpy.

In [ ]:
from train_probe import binary_metrics, stratified_split, train_head

train_idx, val_idx, test_idx = stratified_split(labels, SEED)
print(f"split: {len(train_idx)} train / {len(val_idx)} val / {len(test_idx)} test")


def fmt(value):
    return "n/a" if value is None else f"{value:.3f}"


heads, results = {}, {}
for architecture in ("linear", "mlp"):
    print(f"\n{architecture} probe")
    heads[architecture] = head = train_head(
        features[train_idx],
        y[train_idx],
        features[val_idx],
        y[val_idx],
        architecture=architecture,
        hidden_dim=64,
        epochs=EPOCHS,
        lr=1e-3,
        batch_size=64,
        seed=SEED,
    )
    with torch.no_grad():
        scores = torch.sigmoid(head(features)).tolist()
    results[architecture] = {
        split: binary_metrics([labels[i] for i in idx], [scores[i] for i in idx], threshold=0.5)
        for split, idx in (("train", train_idx), ("val", val_idx), ("test", test_idx))
    }

print(f"\n{'probe':<8} {'split':<6} {'n':>4} {'accuracy':>9} {'auroc':>7}")
for architecture, splits in results.items():
    for split, m in splits.items():
        print(f"{architecture:<8} {split:<6} {m['n']:>4} {fmt(m['accuracy']):>9} {fmt(m['auroc']):>7}")

Pick the probe to keep by its **validation** metrics; the test split is for one final, honest
number. We continue with the linear probe: it is the usual baseline, it's hard to overfit, and its
weights are a single direction in activation space.

## 6. Save the weights and load them into a fresh probe

There is no probe save/load API in Undercurrent v0.1 (it's on the
[roadmap](https://github.com/wrynx/undercurrent/blob/main/ROADMAP.md)), so we save the probe's state
dict directly with `safetensors`: a plain tensor file that, unlike pickle, can't run code when
loaded. It holds the weights plus the standardisation `mean` and `std`.

The example's `save_probe` / `load_probe` in `linear_probe.py` wrap the same calls and also write a
`probe.json` with the layer, tensor type and threshold; use them when you want a self-describing
probe directory.

In [ ]:
from linear_probe import ProbeHead
from safetensors.torch import load_file, save_file

head = heads["linear"]
WEIGHTS = Path("my_probe.safetensors")
save_file({name: t.detach().cpu().contiguous() for name, t in head.state_dict().items()}, str(WEIGHTS))
print(f"saved {WEIGHTS} ({WEIGHTS.stat().st_size} bytes)")

state = load_file(str(WEIGHTS))
print({name: tuple(t.shape) for name, t in state.items()})

fresh = ProbeHead(input_dim=features.shape[1], architecture="linear")
fresh.load_state_dict(state, strict=True)
fresh.eval()

# Round trip: identical tensors, identical scores.
assert all(torch.equal(state[name], t) for name, t in head.state_dict().items())
with torch.no_grad():
    assert torch.equal(fresh(features), head(features))
print("round trip OK: the reloaded probe matches the trained one exactly")

## 7. Run the trained probe live

`TrainedLinearProbe` (from `linear_probe.py`) wraps the head as an Undercurrent probe: for each
activation it computes `score = sigmoid(head(activation))`, and a score at or above `threshold`
returns an `ABORT` signal. Because the extraction point runs **inline** (the default), that signal
stops generation. `spec_for` builds an extraction point at the same layer, tensor type and position
the probe was trained on; reading anything else would make the scores meaningless.

The probe judges the last prompt token. With the Hugging Face backend, an abort there still lets
the first generated token through, because Transformers checks its stopping criteria after each
token.

In [ ]:
from linear_probe import PROBE_TYPE, TrainedLinearProbe, spec_for

from undercurrent.core import ProbeFactory
from undercurrent.model import ProbedModel

factory = ProbeFactory(TrainedLinearProbe, {"head": fresh, "threshold": 0.5, "label_names": label_names})
spec = spec_for({"layers": [LAYER], "tensor_type": "residual_stream", "position": POSITION})
print(spec)

prompts = [
    "Thanks for sharing, the recipe looks great.",
    "Could you tell me more about the new library?",
    "Shut up, you are a worthless idiot.",
    "Only a pathetic moron would believe this garbage.",
]
with ProbedModel.from_pretrained(MODEL, spec=spec, probes={PROBE_TYPE: factory}, device=DEVICE) as model:
    for out in model.generate(prompts, max_new_tokens=20, temperature=0.0):
        verdict = out.probe_results["trained_probe"].verdict
        status = "ABORTED" if out.aborted else f"ok: {out.text!r}"
        print(f"{out.prompt!r}\n  score={verdict['score']:.3f} label={verdict['label']} -> {status}")

The same probe can also run in **observe mode**: add `execution_mode: async` to the extraction
point and attach a log sink, and it scores every request without ever stopping one. See the
[quickstart notebook](https://github.com/wrynx/undercurrent/blob/main/notebooks/quickstart.ipynb).

## 8. Caveats

- **This is a toy probe.** A few hundred examples, a small model and a few seconds of training
  show that the pipeline works; they don't make a toxicity detector. The numbers above say nothing
  about how well a probe would do on real traffic.
- To build something you'd rely on: use a larger and more representative dataset, a stronger model,
  sweep layers and compare validation AUROC, choose the threshold on held-out data for the error
  rate you can accept, and test on data from where the probe will run.
- A probe only means something on the base model, layer, tensor type and position it was trained
  on. A different model with the same hidden size won't raise an error, but its scores are noise.
- Crowd-sourced toxicity labels carry their raters' biases, and the probe learns them too.
- Read [intended use and limitations](https://wrynx.github.io/undercurrent/about/intended-use/)
  before you put a probe in front of users; it includes a "probe card" checklist for documenting
  one.

## Next steps

- Bigger runs on a GPU: the command-line script in
  [`examples/train_probe/`](https://github.com/wrynx/undercurrent/tree/main/examples/train_probe)
  (`train_probe.py --dataset civil_comments --limit 4000 --device cuda`) and `use_probe.py`.
- How probes work: [probes and their lifecycle](https://wrynx.github.io/undercurrent/concepts/probes/)
  and [writing a custom probe](https://wrynx.github.io/undercurrent/guides/custom-probe/).
- Source, issues and discussions: [github.com/wrynx/undercurrent](https://github.com/wrynx/undercurrent).